<a href="https://colab.research.google.com/github/henriquers-dev/arquiteturaTransformer.ipynb/blob/main/arquiteturaTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ARQUITETURA TRANSFORMERS:

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [21]:
class ScaledDotProductAttention(nn.Module):
  def __init__(self, d_k):
    super().__init__()
    self.d_k = d_k # Dimensão das Keys e Queries

  def forward(self, Q, K, V, mask=None):
    # Q, K e V têm formas específicas (batch_size, seq_len, d_k)
    # Calculando pontuação de atenção:
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))

    if mask is not None:
      # o mask pode ter forma de broadcast:
      scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim = -1)
    output = torch.matmul(attn, V) # Forma (batch_size, seq_len, d_k)
    return output, attn

# Aplicação de uso dos transformers:

In [25]:
# Exemplo de uso:
batch_size = 2
seq_len = 4
d_model = 8 # embedding / dimensão do modelo

# Suponha d_k como d_model para simplicidade
d_k = d_model

# Entrada fictícia
x = torch.rand(batch_size, seq_len, d_model)

# Matrizes de projeção
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(x)
K = W_K(x)
V = W_V(x)

# Instanciando a camada de atenção
attention = ScaledDotProductAttention(d_k)
output, attn_weights = attention(Q, K, V)

print("Saída da camada de atenção:", output.shape) #
print("Pesos de atenção:", attn_weights.shape)

Saída da camada de atenção: torch.Size([2, 4, 8])
Pesos de atenção: torch.Size([2, 4, 4])


# Colocando os Multiheads de Atenção

In [28]:
class MultiHeadSelfAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.W_Q = nn.Linear(d_model, d_model)
    self.W_K = nn.Linear(d_model, d_model)
    self.W_V = nn.Linear(d_model, d_model)
    self.W_O = nn.Linear(d_model, d_model)

  def forward(self, x, mask=None):
    batch_size = x.size(0)

    # Projetando Q, K e V
    Q = self.W_Q(x)
    K = self.W_K(x)
    V = self.W_V(x)
     # (batch_size, seq_len, d_model)

    # Dividindo em Heads
    Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

# Criando e executando as MultiHeads

In [45]:
class MultiHeadSelfAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads

    self.W_Q = nn.Linear(d_model, d_model)
    self.W_K = nn.Linear(d_model, d_model)
    self.W_V = nn.Linear(d_model, d_model)
    self.W_O = nn.Linear(d_model, d_model)

  def forward(self, x, mask=None):
    batch_size = x.size(0)
    seq_len = x.size(1) # Get seq_len from input x

    # Projetando Q, K e V
    Q = self.W_Q(x)
    K = self.W_K(x)
    V = self.W_V(x)

    # Dividindo em Heads
    Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    # Atenção por cabeças:
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))

    if mask is not None:
      scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
    attn = F.softmax(scores, dim = -1)
    out = torch.matmul(attn, V) # (batch_size, num_heads, seq_len, d_k)

    # Concatenando cabeças:
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model) # Use self.d_model
    output = self.W_O(out)
    return output, attn

# Testando a multiHead
x = torch.rand(batch_size, seq_len, d_model)
multihead_attention = MultiHeadSelfAttention(d_model, num_heads=2)
output, attn_weights = multihead_attention(x)

print("Saída MultiHeadSelfAttention:", output.shape)
print("Pesos de atenção MultiHeadSelfAttention:", attn_weights.shape)

Saída MultiHeadSelfAttention: torch.Size([2, 4, 8])
Pesos de atenção MultiHeadSelfAttention: torch.Size([2, 2, 4, 4])
